#### Import

In [17]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent

SRC_PATH = PROJECT_ROOT/ 'src'
if str(SRC_PATH) not in sys.path: 
    sys.path.append(str(SRC_PATH))

import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from config import (
    PREPROCESSING_DIR,
    PA_TABLE_PARQUET,
    AREA_LOAD_PARQUET,
    INTERTIES_HOUR_AHEAD_PARQUET,
    INTERTIE_CAPABILITY_PARQUET,
    GENERATION_PARQUET,
    OUTAGES_PARQUET,
    RANDOM_SEED,
)

# Suppress warnings so the outputs in the notebooks stay cleaner. 
warnings.filterwarnings('ignore')
# This handles that annoying output compression where you get '...'
# instead of displaying all columns.
pd.set_option('display.max_columns', None)
# Changes the output width of the jupyter notebook. 
pd.set_option('display.width', 120)
np.random.seed(RANDOM_SEED)

pa_hourly = pd.read_parquet(PA_TABLE_PARQUET)
area_load = pd.read_parquet(AREA_LOAD_PARQUET)
interties_hour_ahead = pd.read_parquet(INTERTIES_HOUR_AHEAD_PARQUET)
intertie_capability = pd.read_parquet(INTERTIE_CAPABILITY_PARQUET)
generation = pd.read_parquet(GENERATION_PARQUET)
outages = pd.read_parquet(OUTAGES_PARQUET)

#### Dataset Overview

In [34]:
summary = []

datasets = {
    'pa_hourly': pa_hourly,
    'area_load': area_load,
    'interties': interties_hour_ahead,
    'intertie_capability': intertie_capability,
    'generation': generation,
    'outages': outages,
}

for name, df in datasets.items():
    summary.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Start': df['timestamp_utc'].min(),
        'End': df['timestamp_utc'].max(),
        'Duplicate timestamps': df['timestamp_utc'].duplicated().sum(),
        'Missing values': int(df.isna().sum().sum()),
        'Memory (MB)': round(df.memory_usage(deep=True).sum() / 1e6, 2),
    })

summary = pd.DataFrame(summary)

summary

,Dataset,Rows,Columns,Start,End,Duplicate timestamps,Missing values,Memory (MB)
0,pa_hourly,100055,5,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0,0,4.00
1,area_load,122736,52,2011-01-01 07:00:00+00:00,2025-01-01 06:00:00+00:00,0,0,50.20
2,interties,136583,8,2010-01-01 07:00:00+00:00,2025-08-01 05:00:00+00:00,0,62786,8.74
3,intertie_capability,100033,9,2015-01-01 07:00:00+00:00,2026-05-31 07:00:00+00:00,0,0,7.20
4,generation,100055,61,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0,985430,48.83
5,outages,100033,13,2015-01-01 07:00:00+00:00,2026-05-31 07:00:00+00:00,0,197686,10.40


In [36]:
for name, df in datasets.items():
    missing = df.isna().sum()

    missing = missing[missing > 0]

    print(f'\n{name}')

    if missing.empty: 
        print('No missing values.')
    else: 
        print(missing.sort_values(ascending = False))


pa_hourly
No missing values.

area_load
No missing values.

interties
export_mt                    31391
import_mt                    31391
hour_ahead_price_forecast        4
dtype: int64

intertie_capability
No missing values.

generation
gas_fired_steam_system_available     53352
gas_fired_steam_system_generation    53352
gas_fired_steam_maximum_capacity     53352
gas_fired_steam_system_capacity      53352
gas_fired_steam_total_generation     53352
storage_total_generation             51143
storage_system_capacity              51143
storage_system_available             51143
storage_maximum_capacity             51143
storage_system_generation            51143
dual_fuel_system_available           50135
dual_fuel_system_generation          50135
dual_fuel_total_generation           50135
dual_fuel_system_capacity            50135
dual_fuel_maximum_capacity           50135
solar_total_generation               25656
solar_system_available               25656
solar_system_capacity       

* Hour ahead price forecast only missing 4 values is essentailly complete. 
* Montana may have not been reporting intertie data for parts of the historical record. 
* Generation is a different case, you'd expect missing values as technologies phase-in and phase-out across time.
* Same applies to outages - technologies have to exist to experience outages. 

#### External Data Checks

In [ ]:
print(interties_hour_ahead.loc[
    interties_hour_ahead['import_mt'].notna(),
    'timestamp_utc'
].min())

print(interties_hour_ahead.loc[
    interties_hour_ahead['export_mt'].notna(),
    'timestamp_utc'
].min())

2013-08-01 06:00:00+00:00
2013-08-01 06:00:00+00:00


In [42]:
cutoff = pd.Timestamp('2013-08-01 06:00:00+00:00', tz = 'UTC')

mt_missing = (
    interties_hour_ahead.loc[interties_hour_ahead['timestamp_utc'] > cutoff,
        ['import_mt', 'export_mt']].isna().sum()
)

print(mt_missing)

import_mt    0
export_mt    0
dtype: int64


Montana Intertie Data: 
* import_mt and export_mt are missing prior to August 2013. Investigation confirmed these missing values are expected: the Montana-Alberta Tie Line (MATL) entered service in 2013, and AESO public reporting of Montana intertie schedules and transfer capability began around the commissioning period. These missing values therefore represent the absence of an operational/reporting intertie rather than data quality issues. Missing values disappear after 2013-08-01 06:00:00+00:00 UTC. 

In [43]:
generation_coverage = []

for col in generation.columns:
    if col == 'timestamp_utc': 
        continue

    non_missing = generation.loc[generation[col].notna(), 'timestamp_utc']

    generation_coverage.append({
        'Column': col,
        'First non-missing': non_missing.min(),
        'Last non-missing': non_missing.max(),
        'Missing': generation[col].isna().sum(),
    })

generation_coverage = (
    pd.DataFrame(generation_coverage).sort_values('First non-missing')
)

generation_coverage

,Column,First non-missing,Last non-missing,Missing
0,coal_system_generation,2015-01-01 07:00:00+00:00,2024-07-01 05:00:00+00:00,16800
28,other_system_available,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
58,total_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
32,wind_system_available,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
33,coal_system_capacity,2015-01-01 07:00:00+00:00,2024-07-01 05:00:00+00:00,16800
34,cogeneration_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
35,combined_cycle_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
38,hydro_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
39,other_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0
40,simple_cycle_system_capacity,2015-01-01 07:00:00+00:00,2026-06-01 05:00:00+00:00,0


Generation Data: 
* Missingness in the generation dataset is highly structured by fuel category. Coal and dual-fuel fields terminate around Alberta’s final coal-to-gas conversions in July 2024, while solar and storage fields begin later as those categories entered AESO reporting. Gas-fired steam and the beginning of dual-fuel coverage likely reflect AESO classification or reporting changes rather than the physical introduction of those technologies. Because all metrics within each category share identical coverage boundaries, the missing values are treated as structurally unavailable rather than randomly missing.
